In [2]:
import pandas as pd
from pathlib import Path

root = Path("./output")

entities = pd.read_parquet(root / "entities.parquet")
relationships = pd.read_parquet(root / "relationships.parquet")
communities = pd.read_parquet(root / "communities.parquet")
community_reports = pd.read_parquet(root / "community_reports.parquet")
text_units = pd.read_parquet(root / "text_units.parquet")
documents = pd.read_parquet(root / "documents.parquet")

## Entity diagnostics

In [3]:
entities.head()

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree
0,54c5690a-ef57-43d7-a625-de03dadba989,0,DECEMBER,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,2,30
1,c482e454-f621-447d-ba64-7fbbb7123314,1,CHRISTMAS PRESENT,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,6,17
2,ebd02c68-b678-4f27-9118-fec281c4e782,2,MAY,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,6,19
3,0ec8f093-4cf5-47f3-858c-83a11d2d9886,3,THE FIRST OF THE THREE SPIRITS,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,2,33
4,34b1c878-ada0-4f8e-9d2c-607a8152b612,4,RESTRICTIONS WHATSOEVER,NOUN PHRASE,,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...,3,35


In [4]:
# GraphRAG entity outputs commonly include fields like name, type, degree, and frequency, so those are the first columns to profile.
entity_cols = [c for c in ["title", "type", "degree", "frequency", "description"] if c in entities.columns]
entity_profile = entities[entity_cols].copy()

if "degree" in entity_profile.columns:
    entity_profile["degree"] = pd.to_numeric(entity_profile["degree"], errors="coerce")
if "frequency" in entity_profile.columns:
    entity_profile["frequency"] = pd.to_numeric(entity_profile["frequency"], errors="coerce")

top_entities = entity_profile.sort_values(
    [c for c in ["degree", "frequency"] if c in entity_profile.columns],
    ascending=False
).head(50)

top_entities

,title,type,degree,frequency,description
5,GHOST,NOUN PHRASE,79,25,
28,TINY TIM,NOUN PHRASE,47,11,
20,LACE TUCKER,NOUN PHRASE,45,4,
84,SPIRIT,NOUN PHRASE,44,18,
24,JOE,NOUN PHRASE,44,4,
26,BOISTEROUS GROUP,NOUN PHRASE,42,4,
27,ILLUSTRATION,NOUN PHRASE,41,9,
173,COPYRIGHT LAW,NOUN PHRASE,40,4,
35,BOB CRATCHIT,NOUN PHRASE,36,9,
10,LADEN,NOUN PHRASE,36,2,


## Relationship diagnostics

In [5]:
relationships.head()

,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,03535beb-5131-410c-abc4-683a467848b7,0,BOB CRATCHIT,CAROLINE,,0.000201,65,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
1,9c5578bf-95df-41c0-956e-215dcba5c927,1,BOB CRATCHIT,CHRISTMAS,,0.000281,64,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
2,bd3c51be-2fb2-455e-b42c-017888f065e2,2,BOB CRATCHIT,CHRISTMAS CAROL,,0.000163,71,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
3,1269b96b-14c2-46a2-9b9a-95f019d9a3db,3,BOB CRATCHIT,CHRISTMAS PRESENT,,0.000327,53,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...
4,12597e32-c94c-4240-b7b9-6f41dfca9e3f,4,BOB CRATCHIT,FRED,,0.000117,53,[9cdcd505919ac2a29e3f977f81b358ac7f2870c82cb65...


In [6]:
rel_cols = [c for c in ["source", "target", "weight", "description"] if c in relationships.columns]
rel_profile = relationships[rel_cols].copy()

if "weight" in rel_profile.columns:
    rel_profile["weight"] = pd.to_numeric(rel_profile["weight"], errors="coerce")

top_relationships = rel_profile.sort_values(
    [c for c in ["weight"] if c in rel_profile.columns],
    ascending=False
).head(50)

top_relationships

,source,target,weight,description
395,GHOST,SPIRIT,0.000864,
527,BOB,TINY TIM,0.000686,
11,BOB CRATCHIT,TINY TIM,0.000686,
410,BOB,BOB CRATCHIT,0.000622,
199,FRED,PLUMP SISTER,0.000595,
61,CHRISTMAS,GHOST,0.000578,
429,CHRISTMAS,SPIRIT,0.000562,
970,ELECTRONIC WORKS,PROJECT GUTENBERG LITERARY ARCHIVE,0.000550,
968,ELECTRONIC WORKS,PROJECT GUTENBERG,0.000509,
1020,PROJECT GUTENBERG,PROJECT GUTENBERG LITERARY ARCHIVE,0.000509,


## Community hierarchy

In [7]:
communities.head()

,id,human_readable_id,community,level,parent,children,title,entity_ids,relationship_ids,text_unit_ids,period,size
0,36d40a0b-55b6-4cc6-a7cd-963fc2fbc089,0,0,0,-1,"[8, 9]",Community 0,"[66eb8407-b428-461a-a0f3-42ade3d46679, bae3ad7...","[005c74fa-b6ba-4bf4-acfd-9431536ab192, 00e40bb...",[36309eb460a63b78dbfb15b2448081485eee7634b1ebb...,2026-06-21,28
1,9f8e6ce0-70fd-4c78-8c5c-0d86d88c5175,1,1,0,-1,"[10, 11]",Community 1,"[fe09a8e2-3c4b-4a90-b30b-cf3d90b0b44f, a5e1f1a...","[01cbc5f8-cab7-412c-9ab6-ff1fe51a30b0, 03791c8...",[1092b8131a142ed2e9661cbc9064dd521429b46505d4b...,2026-06-21,14
2,67058e39-e2b8-4ee9-8404-0bd4b4b729fc,2,2,0,-1,"[12, 13, 14, 15]",Community 2,"[90cb8721-deb7-4664-8fa0-1fd72437b6e9, a633b4d...","[002d3920-456b-4548-9865-d6977ab2a3f2, 0037434...",[007c10bf87800ae76ecd2d293198b036d4bdfc97c79d7...,2026-06-21,42
3,61749a9f-942e-4cec-b7fd-5b4e3c2c8332,3,3,0,-1,"[16, 17]",Community 3,"[d4465f84-816d-4919-89a2-ca586d18e25c, 1892365...","[01ba0641-5926-497c-98c3-bc4709b00a2a, 025a86c...",[153a5b3e8f4371e00bf58b73dfa346af552b09389be54...,2026-06-21,24
4,4cf83144-7304-4895-8775-29e5bcd51143,4,4,0,-1,"[18, 19]",Community 4,"[228a8926-0c2d-43be-ae0d-d167ae8738ac, 1322635...","[0019dbdc-0c96-440e-957a-b0fab05a674a, 0593bd3...",[332deb9b6b27ca253056816eb64198676c9ba75f952fe...,2026-06-21,20


In [7]:
#GraphRAG communities are hierarchical, so community, parent, children, level, and size are useful for checking whether clustering looks coherent.
comm_cols = [c for c in ["community", "parent", "children", "level", "title", "size"] if c in communities.columns]
comm_profile = communities[comm_cols].copy()

if "level" in comm_profile.columns:
    comm_profile["level"] = pd.to_numeric(comm_profile["level"], errors="coerce")
if "size" in comm_profile.columns:
    comm_profile["size"] = pd.to_numeric(comm_profile["size"], errors="coerce")

community_summary = comm_profile.sort_values(
    [c for c in ["level", "size"] if c in comm_profile.columns],
    ascending=[True, False][:len([c for c in ["level", "size"] if c in comm_profile.columns])]
)

community_summary.head(50)

,community,parent,children,level,title,size
2,2,-1,"[12, 13, 14, 15]",0,Community 2,42
6,6,-1,"[23, 24, 25, 26]",0,Community 6,34
0,0,-1,"[8, 9]",0,Community 0,28
5,5,-1,"[20, 21, 22]",0,Community 5,25
3,3,-1,"[16, 17]",0,Community 3,24
4,4,-1,"[18, 19]",0,Community 4,20
1,1,-1,"[10, 11]",0,Community 1,14
7,7,-1,"[27, 28, 29]",0,Community 7,14
8,8,0,"[30, 31]",1,Community 8,17
15,15,2,"[35, 36, 37]",1,Community 15,17


## Community reports review

In [10]:
community_reports.head()

,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
0,4f63a992d87729a065216f9e06f6f3256921ab62646afb...,33,33,2,12,[],A Christmas Carol Character Analysis as of 1843,This report delves into the key characters of ...,# A Christmas Carol Character Analysis as of 1...,8.5,The moderate to high importance rating reflect...,"[{'explanation': 'The protagonist, Ebenezer Sc...","{\n ""title"": ""A Christmas Carol Character A...",2026-06-21,3
1,75d01e439200ef1f12848511c60e280e0e3a733783cbd2...,46,46,2,19,[],The Isolated World of Ebenezer Scrooge,This report provides an overview of Ebenezer S...,# The Isolated World of Ebenezer Scrooge\n\nTh...,8.5,The importance rating reflects the significant...,[{'explanation': 'Ebenezer Scrooge is portraye...,"{\n ""title"": ""The Isolated World of Ebeneze...",2026-06-21,5
2,d922ffc9c5967c272742566bcd6e33e0b90041d35cf323...,47,47,2,24,[],A Christmas Carol: Scrooge's Encounter with Ma...,The report delves into Ebenezer Scrooge's enco...,# A Christmas Carol: Scrooge's Encounter with ...,8.5,The high importance rating reflects the pivota...,[{'explanation': 'The encounter between Scroog...,"{\n ""title"": ""A Christmas Carol: Scrooge's ...",2026-06-21,3
3,23dbf4ced82b19567224c0f3f56ec184fce9ae26ba827b...,9,9,1,0,[],Project Gutenberg Literary Archive Foundation ...,This report provides an overview of the Projec...,# Project Gutenberg Literary Archive Foundatio...,8.5,The high importance rating reflects the critic...,[{'explanation': 'The Project Gutenberg Litera...,"{\n ""title"": ""Project Gutenberg Literary Ar...",2026-06-21,11


In [9]:
# Community reports are often the most useful layer for evaluating whether the graph abstraction is actually informative, 
# since they contain LM-generated findings and summaries.
report_cols = [c for c in ["community", "level", "title", "rank", "summary", "findings"] if c in community_reports.columns]
report_profile = community_reports[report_cols].copy()

if "rank" in report_profile.columns:
    report_profile["rank"] = pd.to_numeric(report_profile["rank"], errors="coerce")

top_reports = report_profile.sort_values(
    [c for c in ["rank"] if c in report_profile.columns],
    ascending=False
).head(20)

In [33]:
for row in top_reports[["title", "summary"]].itertuples(index=False, name=None):
    print(row)
    print()

('A Christmas Carol Character Analysis as of 1843', "This report delves into the key characters of Charles Dickens' A Christmas Carol, illustrating how these entities interact within the narrative framework to convey themes of redemption and personal growth. The information is relevant to the story's climax around Christmas time in the 1840s.")

('The Isolated World of Ebenezer Scrooge', "This report provides an overview of Ebenezer Scrooge's character and his interactions with others, highlighting his isolated nature and aversion to social sympathy, as demonstrated through his behavior on Christmas Eve. The information is relevant to understanding Scrooge's personality and behavior around the Christmas period. ")

("A Christmas Carol: Scrooge's Encounter with Marley's Ghost", "The report delves into Ebenezer Scrooge's encounter with Marley's Ghost, exploring how the entities interact within the narrative framework to convey the themes of redemption and the supernatural. The informatio